In [ ]:
import amici
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)

# Use all available CPU cores for AMICI model compilation
os.environ["AMICI_PARALLEL_COMPILE"] = ""

# First steps in AMICI

![](https://amici.readthedocs.io/en/latest/_images/amici_objects.png)

* [API doc](https://amici.readthedocs.io/en/latest/python_modules.html)
* [Further examples](https://amici.readthedocs.io/en/latest/python_examples.html)


## SBML import

In [ ]:
from amici.importers.sbml import SbmlImporter

sbml_file = "petab_problem/model.xml"
amici_model = SbmlImporter(sbml_file).sbml2amici(model_name="conversion1")

In [ ]:
print(amici_model.get_free_parameter_ids())

In [ ]:
# two state variables: A and B
print(amici_model.get_state_ids())

In [ ]:
# by default, all species and compartments are treated as observables
print(amici_model.get_observable_ids())

## Model simulation & ReturnData

In [ ]:
amici_model.set_timepoints(np.linspace(0, 10, 101))
rdata = amici_model.simulate()
rdata

In [ ]:
from amici.sim.sundials.plotting import plot_state_trajectories

plot_state_trajectories(rdata)

#### ReturnData

`ReturnData` / `ReturnDataView` is the main data structure for simulation results in AMICI. It contains the state trajectories, observable trajectories, sensitivities, likelihoods, and more.

In [ ]:
for attr in dir(rdata):
    if not attr.startswith("_"):
        print(attr)

The most important attributes for our purposes are:
* $x$: state trajectories
* $y$: observable trajectories
* $w$: various model expressions, e.g. fluxes, assignment rules, ...

In [ ]:
# 'x': the state trajectories
rdata.x

In [ ]:
# More conveniently, as xarray.DataArray via the `xr` attribute
rdata.xr.x.head()

In [ ]:
# select only species B
rdata.xr.x.loc[:, "B"].head()

In [ ]:
rdata.xr.x.to_dataframe().head()

In [ ]:
# 'w': various model expressions, e.g. fluxes, assignment rules, ...
rdata.xr.w.to_dataframe().head()

In [ ]:
plt.plot(rdata.xr.w.time, rdata.xr.w.loc[:, "flux_R_ab"] - rdata.xr.w.loc[:, "flux_R_ba"])
plt.xlabel("time")
plt.ylabel("net flux A→B");

In [ ]:
# for interactive exploration, we can also directly access many model quantities by their IDs via `ReturnData.by_id`
rdata.by_id("A")

In [ ]:
# We can simplify visualizing the net flux further:
from amici.sim.sundials.plotting import plot_expressions

plot_expressions("flux_R_ab - flux_R_ba", rdata)

### Changing parameter values

In [ ]:
amici_model.set_timepoints(np.linspace(0, 100, 101))
amici_model.set_free_parameter_by_id("k_ab", 0.05)
amici_model.set_free_parameter_by_id("k_ba", 0.02)

print("Free parameters:", dict(zip(amici_model.get_free_parameter_ids(), amici_model.get_free_parameters())))

rdata = amici_model.simulate()
plot_state_trajectories(rdata)

In [ ]:
for a0 in [0.5, 1, 2, 4]:
    amici_model.set_free_parameter_by_id("A0", a0)
    rdata = amici_model.simulate()
    plt.plot(rdata.ts, rdata.by_id("flux_R_ab") - rdata.by_id("flux_R_ba"), label=f"A0={a0}")
plt.legend(bbox_to_anchor=(1, 1))
plt.xlabel("time")
plt.ylabel("net flux A→B")
plt.title("Parameter scan");

## Changing Solver options

In the examples above, we used the default solver options. However, AMICI offers a wide range of options for the underlying SUNDIALS solvers, which can be set via the `Solver` object.

In [ ]:
solver = amici_model.create_solver()

print(solver.get_class_name())
solver

In [ ]:
rdata = amici_model.simulate(solver=solver)
rdata

In [ ]:
solver.set_max_steps(10)
rdata = amici_model.simulate(solver=solver)
rdata

In [ ]:
solver.set_max_steps(1000)
rdata = amici_model.simulate(solver=solver)
print(rdata.numsteps)
rdata

In [ ]:
# this is where all sundials-related objects reside
import amici.sim.sundials as asd

# Enable forward sensitivity analysis
solver.set_sensitivity_method(asd.SensitivityMethod.forward)
solver.set_sensitivity_order(asd.SensitivityOrder.first)

rdata = asd.run_simulation(amici_model, solver)

# state sensitivities are available via the `sx` attribute in the `ReturnData` object
rdata.xr.sx.to_dataframe()

In [ ]:
rdata.xr.sx.coords

In [ ]:
for xx in amici_model.get_state_ids():
    for pp in amici_model.get_free_parameter_ids():
        plt.plot(rdata.xr.sx.time, rdata.xr.sx.loc[:, pp, xx], label=f"{xx} w.r.t. {pp}")

plt.xlabel("time")
plt.ylabel(f"sx")
plt.title("State sensitivities")
plt.legend(bbox_to_anchor=(1, 1));

In [ ]:
# there are quite a few options for the solver, which can be set via the `set_*` methods.
# You can find all options in the AMICI documentation
# https://amici.readthedocs.io/en/latest/generated/amici.sim.sundials.html#amici.sim.sundials.Solver
[a for a in dir(solver) if not a.startswith("_")]

`Model.simulate` is good for quick interactive simulations. If you are running large numbers of simulations, `amici.sim.sundials.run_simulation`/`amici.sim.sundials.run_simulations` is slightly more efficient.

In [ ]:
rdata = asd.run_simulation(amici_model, solver)
rdata

## Loading existing models

Avoiding the SBML import step, we can also reuse existing models that have already been compiled:

In [ ]:
model_module = amici.import_model_module("conversion1", "amici_models/1.0.1/conversion1/")
model_module.get_model()

# `conversion1` is the model name we passed to `sbml2amici` during model import
# `amici_models/1.0.1` is the AMICI-version-specific default directory for compiled models.
# The directory can be changed via the `output_dir` argument in the `sbml2amici` method.

## Antimony import
Antimony [documentation](https://tellurium.readthedocs.io/en/latest/antimony.html), [paper](https://doi.org/10.48550/arXiv.2405.15109).

In [ ]:
from amici.importers.antimony import antimony2amici

antimony_model = """
    A0 = 1
    B0 = 0
    species A = A0, B = B0
    R_ab: A -> B; k_ab * A
    R_ba: B -> A; k_ba * B
    k_ab = 0.5
    k_ba = 0.5
"""
amici_model: amici.sim.sundials.Model = antimony2amici(antimony_model, model_name="conversion_from_antimony")


*(When importing several models, make sure to use different model names to avoid name clashes.)*

In [ ]:
amici_model.get_name()

In [ ]:
amici_model.get_state_ids()

## ExpData - representing different experimental setups and computing likelihood

`ExpData` is the main data structure for representing experimental data and conditions in AMICI.
It can be used to specify different experimental setups, e.g. by changing initial conditions or parameter values,
and to compute the likelihood of the data given the model and parameters.

In [ ]:
edata = asd.ExpData(amici_model)

In [ ]:
observable_ids = amici_model.get_observable_ids()
print(observable_ids)

In [ ]:
measurement_timepoints = [1, 2, 3]
measurements_A = [0.8, 0.6, 0.4]
measurements_B = [0.2, 0.4, 0.6]

edata.set_timepoints(measurement_timepoints)
edata.set_measurements(measurements_A, observable_ids.index("yA"))
edata.set_measurements(measurements_B, observable_ids.index("yB"))
edata.set_noise_scales(0.05)

In [ ]:
rdata = asd.run_simulation(amici_model, solver, edata)

In [ ]:
from amici.sim.sundials.plotting import plot_observable_trajectories

plot_observable_trajectories(rdata, edata=edata)
plt.legend(bbox_to_anchor=(1, 1));

In [ ]:
# the log-likelihood of the data given the model and parameters is available via the `llh` attribute
# (by default, independent Gaussian noise is assumed. this can be changed during model import.)
print(rdata.llh)

In [ ]:
# the gradient of the likelihood with respect to the parameters is available via the `sllh` attribute
print(rdata.sllh)

In [ ]:
for attr in dir(edata):
    if not attr.startswith("_"):
        print(attr)

### Creating a custom objective function for pypesto

Now we know how to compute the likelihood and its gradient for a given set of parameters, we can use this to create a custom objective function for pypesto.

In [ ]:
def my_negloglikelihood(x: np.ndarray) -> tuple[float, np.ndarray]:
    # apply parameters to amici model
    amici_model.set_free_parameters(x)

    # run simulation and compute likelihood
    rdata = asd.run_simulation(amici_model, solver, edata)

    # negative log-likelihood and its gradient
    return -rdata.llh, -rdata.sllh

from pypesto import Objective, Problem
from pypesto.optimize import FidesOptimizer, minimize

par_ids = amici_model.get_free_parameter_ids()
obj = Objective(fun=my_negloglikelihood, grad=True, x_names=par_ids)
problem = Problem(objective=obj, lb=[0]*len(par_ids) , ub=[2]*len(par_ids))

optimizer = FidesOptimizer()

result = minimize(problem, optimizer=optimizer, n_starts=10)

In [ ]:
import pandas as pd

pd.DataFrame(result.optimize_result.get_for_key("x"), columns=par_ids)

This works for the purpose of our example, but it's not very efficient and does not support parallelization.
If possible, use pypesto's `PetabImporter` to create a `pypesto.Problem`/`pypesto.Objective` directly from the PEtab problem, which will support parallelization and additional features.

### Accessing AMICI objects in the pypesto-PEtab-workflow

We use pypesto's PEtab import as before:

In [ ]:
from pypesto.petab import PetabImporter

petab_yaml = "petab_problem/problem.yaml"
petab_importer = PetabImporter.from_yaml(petab_yaml)
pypesto_problem = petab_importer.create_problem()
objective = pypesto_problem.objective

In [ ]:
pypesto_problem.x_names

In [ ]:
objective(np.array([0.5, 0.5]))

In [ ]:
# return_dict=True gives us access to the underlying AMICI ReturnData objects
res = objective(np.array([0.5, 0.5]), return_dict=True)
res

In [ ]:
rdata = res['rdatas'][0]
print(f"{rdata.numsteps=}")
print(f"{rdata.cpu_time_total=}ms")

#### Changing solver options

The AMICI solver options can be accessed via the `amici_solver` attribute of the generated objective function.

In [ ]:
objective.amici_solver

In [ ]:
objective.amici_solver.get_absolute_tolerance(), objective.amici_solver.get_relative_tolerance()

In [ ]:
objective.amici_solver.set_absolute_tolerance(1e-10)
objective.amici_solver.set_relative_tolerance(1e-4)

In [ ]:
res = objective(np.array([0.5, 0.5]), return_dict=True)
rdata = res['rdatas'][0]
print(f"{rdata.numsteps=}")
print(f"{rdata.cpu_time_total=}ms")

In [ ]:
# Use adjoint sensitivity analysis instead of the default forward sensitivity analysis.
# This is more efficient for models with many parameters and few state variables.
objective.amici_solver.set_sensitivity_method(asd.SensitivityMethod.adjoint)

objective(np.array([0.5, 0.5]), sensi_orders=(0, 1), return_dict=True)

#### Inspecting the ExpData objects

In [ ]:
# We can also inspect the ExpData object that was used for this simulation
#  (but don't modify it)
objective.edatas

In [ ]:
plt.scatter(objective.edatas[0].get_timepoints(), objective.edatas[0].get_measurements())
plt.xlabel("time")
plt.ylabel("measurement")
plt.ylim(bottom=0)

## Summary

What we have seen in this demo:

* How to import SBML and Antimony models into AMICI
* How to simulate models and access the results via the `ReturnData` object
* How to change parameter values and solver options
* How to create a custom objective function for pypesto using AMICI
* How to access AMICI objects in the pypesto-PEtab workflow

![](https://amici.readthedocs.io/en/latest/_images/amici_objects.png)
